# MCIS — Maritime Conflict Intelligence System

## End-to-End Pipeline Notebook

This notebook runs the complete MCIS pipeline: configuration → loading → cleaning → feature engineering → aggregation → analysis → modeling → visualization.

In [2]:
import sys
import os
from pathlib import Path

# Ensure project root is on sys.path so mcis is importable
_nb_dir = Path(os.getcwd()).resolve()
_project_root = _nb_dir.parent if _nb_dir.name == "notebook" else _nb_dir
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
os.chdir(str(_project_root))
print(f"Project root: {_project_root}")

from __future__ import annotations
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import yaml

import mcis.compat  # noqa: F401  (NumPy/statsmodels compat)

from mcis.loader import AISLoader
from mcis.cleaner import AISCleaner
from mcis.features import AISFeatureEngineer
from mcis.aggregator import AISAggregator, AGG_SPEC
from mcis.config_schema import validate_config
from mcis.validation import (
    validate_data_validity_mode, validate_claim_level,
    write_run_metadata, compute_file_hash,
)
from mcis.utils.io import ensure_dir, load_yaml
from mcis.utils.seeds import set_global_seed

print("All imports successful.")

Project root: C:\Users\user\mcis
All imports successful.


---
## 1. Configuration

Load and validate the experiment configuration from `config/settings.yaml`.

In [3]:
CONFIG_PATH = "config/settings.yaml"

# Load raw YAML
with open(CONFIG_PATH) as f:
    raw_cfg = yaml.safe_load(f)

# Validate with Pydantic schema
cfg = validate_config(raw_cfg, defaults_path=CONFIG_PATH)

# Validate guardrails
validate_data_validity_mode(cfg)
validate_claim_level(cfg)
set_global_seed(cfg.get("project", {}).get("random_seed", 42))

print(f"Project: {cfg['project']['name']}")
print(f"Mode: {cfg['project']['data_validity_mode']}")
print(f"Claim level: {cfg['project']['claim_level']}")
print(f"Conflict date (T0): {cfg['conflict']['t0']}")
print(f"Zone: {cfg['conflict']['zone_name']}")

Project: mcis
Mode: empirical
Claim level: predictive_prototype
Conflict date (T0): 2022-02-24
Zone: Black Sea / Sea of Azov


---
## 2. Data Loading

Use `AISLoader` to load raw AIS CSV files with schema enforcement and UTC timestamp parsing.

In [4]:
DATA_DIR = Path(cfg["data"]["raw_dir"])
PRIMARY_FILE = DATA_DIR / cfg["data"]["primary_file"]
DEV_FILE = DATA_DIR / cfg["data"]["dev_file"]

# Use the smallest file for quick demonstration
if (DATA_DIR / "ais_blacksea_3d.csv").exists():
    file_path = str(DATA_DIR / "ais_blacksea_3d.csv")
elif DEV_FILE.exists():
    file_path = str(DEV_FILE)
else:
    file_path = str(PRIMARY_FILE)

print(f"Loading: {file_path}")

loader = AISLoader(cfg)
t0 = time.time()
df_raw, load_report = loader.load_with_report(file_path)
print(f"Loaded {len(df_raw)} rows in {time.time() - t0:.1f}s")
print(f"Date range: {load_report['date_min']} to {load_report['date_max']}")
print(f"Unique MMSI: {load_report['unique_mmsi']}")
print(f"Unique vessel types: {load_report['unique_vessel_types']}")
print(f"Columns: {list(df_raw.columns)}")

Loading: data\raw\ais_blacksea_3d.csv
Loaded 764 rows in 0.0s
Date range: 2022-02-23 00:01:59+00:00 to 2022-02-25 23:48:53+00:00
Unique MMSI: 687
Unique vessel types: 74
Columns: ['vesselUid', 'mmsi', 'imo', 'longitude', 'latitude', 'sog', 'cog', 'rot', 'heading', 'navStatus', 'posMsgType', 'posSrc', 'vesselName', 'callsign', 'flag', 'vesselTypeAis', 'vesselType', 'length', 'width', 'dwt', 'grt', 'destination', 'eta', 'draught', 'staticMsgType', 'staticSrc', 'posDt', 'staticDt', 'insertDt', 'date', 'days_to_t0']


---
## 3. Data Cleaning

Apply 10 sequential cleaning steps via `AISCleaner`: coordinate filter, SOG filter, ROT/heading/COG normalization, navStatus, dimensions, dedup, min-obs filter, and quality flags.

In [5]:
cleaner = AISCleaner(cfg)
t0 = time.time()
df_clean = cleaner.clean(df_raw)
print(f"Cleaned {len(df_clean)} rows in {time.time() - t0:.1f}s")

clean_report = cleaner.cleaning_report()
dropped = clean_report.get("total_initial", {}).get("after", 0) - clean_report.get("total_final", {}).get("after", 0)
print(f"Dropped: {dropped} records")

for step, counts in clean_report.items():
    if isinstance(counts, dict) and "before" in counts:
        print(f"  {step}: {counts['before']} -> {counts['after']}")

Cleaned 9 rows in 0.0s
Dropped: 755 records
  coordinate_filter: 764 -> 705
  sog_filter: 705 -> 705
  rot_normalize: 705 -> 705
  heading_normalize: 705 -> 705
  cog_normalize: 705 -> 705
  navstatus_normalize: 705 -> 705
  dimension_normalize: 705 -> 705
  dedup: 705 -> 705
  min_obs_filter: 705 -> 9
  quality_flags: 9 -> 9
  total_initial: 764 -> 764
  total_final: 764 -> 9


---
## 4. Feature Engineering

Create row-level, trajectory, and rolling features via `AISFeatureEngineer`.

In [6]:
engineer = AISFeatureEngineer(cfg)
t0 = time.time()
df_feat = engineer.transform(df_clean)
print(f"Engineered {len(df_feat)} rows x {len(df_feat.columns)} cols in {time.time() - t0:.1f}s")

# Show key feature columns
feat_cols = [
    "speed_state", "rot_spike", "vessel_category", "flag_group",
    "time_gap_hours", "ais_silence", "step_dist_km", "implied_speed_kt",
    "cog_change", "turn_event",
    "sog_rolling_mean_7d", "sog_rolling_std_7d",
]
available = [c for c in feat_cols if c in df_feat.columns]
print("\nSample features:")
df_feat[available].head(3)

Engineered 9 rows x 58 cols in 0.1s

Sample features:


,speed_state,rot_spike,vessel_category,flag_group,time_gap_hours,ais_silence,step_dist_km,implied_speed_kt,cog_change,turn_event,sog_rolling_mean_7d,sog_rolling_std_7d
0,stopped,False,other,nato,NaN,False,0.000000,NaN,NaN,False,0.0,NaN
1,stopped,False,other,nato,14.622500,False,0.024986,0.000923,140.6,True,0.0,0.0
2,stopped,False,other,nato,30.903056,False,2.629430,0.045943,136.1,True,0.0,0.0


---
## 5. Aggregation

Aggregate vessel-level data into spatial-grid and whole-basin daily panels via `AISAggregator`.

In [7]:
aggregator = AISAggregator(cfg)
t0 = time.time()
panels = aggregator.transform(df_feat)
print(f"Aggregated in {time.time() - t0:.1f}s")

grid_panel = panels["grid_daily"]
blacksea_panel = panels["blacksea_daily"]

print(f"Grid panel: {grid_panel.shape} with {grid_panel.index.nlevels} index levels")
print(f"Black Sea panel: {blacksea_panel.shape}")
print(f"\nBlack Sea panel columns:\n{list(blacksea_panel.columns)}")
blacksea_panel.head(5)

Aggregated in 0.1s
Grid panel: (6, 19) with 2 index levels
Black Sea panel: (3, 23)

Black Sea panel columns:
['vessel_count', 'unique_mmsi', 'mean_sog', 'std_sog', 'mean_rot_abs', 'max_abs_rot', 'rot_spike_count', 'rot_spike_fraction', 'ais_silence_count', 'ais_silence_fraction', 'cog_variance', 'mean_turn_events', 'mean_draught', 'mean_draught_fraction', 'mean_speed_discrepancy', 'max_speed_discrepancy', 'sat_src_fraction', 'no_destination_fraction', 'russian_flag_fraction', 'nato_flag_fraction', 'route_entropy', 'post_conflict', 'days_to_t0']


,vessel_count,unique_mmsi,mean_sog,std_sog,mean_rot_abs,max_abs_rot,rot_spike_count,rot_spike_fraction,ais_silence_count,ais_silence_fraction,...,mean_draught_fraction,mean_speed_discrepancy,max_speed_discrepancy,sat_src_fraction,no_destination_fraction,russian_flag_fraction,nato_flag_fraction,route_entropy,post_conflict,days_to_t0
time_bucket,,,,,,,,,,,,,,,,,,,,,
2022-02-23,5,3,0.02,0.044721,0.092,0.19,0,0.0,0,0.0,...,0.0,0.007570,0.014217,0.4,0.2,0.4,0.6,1.921928,0,-1
2022-02-24,2,1,0.00,0.000000,0.190,0.36,0,0.0,0,0.0,...,0.0,0.143048,0.181208,0.0,1.0,0.0,1.0,NaN,1,0
2022-02-25,2,2,0.00,0.000000,0.145,0.27,0,0.0,1,0.5,...,NaN,0.028712,0.045943,0.5,0.0,0.5,0.5,NaN,1,1


---
## 6. Visual Exploration

Plot before/after time series and a multi-metric panel for key aggregate metrics.

In [8]:
try:
    from mcis.viz.timeseries import plot_before_after, plot_multi_metric_panel, plot_flag_composition
    from mcis.viz.heatmaps import plot_correlation_heatmap
    _viz_ok = True
except ImportError as _e:
    _viz_ok = False
    print(f"Viz imports unavailable: {_e}")

OUTPUT_DIR = Path("outputs")
FIG_DIR = OUTPUT_DIR / "figures"
ensure_dir(FIG_DIR)

t0_date = cfg["conflict"]["t0"]

if _viz_ok:
    fig = plot_before_after(blacksea_panel, "vessel_count", FIG_DIR / "vessel_count_before_after.png", t0=t0_date)
    print("Saved: vessel_count_before_after.png")

    top_metrics = ["vessel_count", "mean_sog", "std_sog", "ais_silence_count", "route_entropy", "cog_variance"]
    available_metrics = [m for m in top_metrics if m in blacksea_panel.columns]
    fig = plot_multi_metric_panel(blacksea_panel, available_metrics, FIG_DIR / "multi_metric_panel.png", t0=t0_date)
    print("Saved: multi_metric_panel.png")

    flag_cols = [c for c in ["russian_flag_fraction", "ukrainian_flag_fraction", "nato_flag_fraction"] if c in blacksea_panel.columns]
    if len(flag_cols) >= 2:
        fig = plot_flag_composition(blacksea_panel, FIG_DIR / "flag_composition.png", t0=t0_date, flag_cols=flag_cols)
        print("Saved: flag_composition.png")

    fig = plot_correlation_heatmap(blacksea_panel, FIG_DIR / "correlation_heatmap.png", t0=t0_date)
    print("Saved: correlation_heatmap.png")
else:
    print("Visualization section skipped (missing optional dependencies)")


Saved: vessel_count_before_after.png
Saved: multi_metric_panel.png
Saved: flag_composition.png
Saved: correlation_heatmap.png


---
## 7. Event Study Analysis

Compute abnormal deviations around T0 using `run_event_study`.

In [9]:
from mcis.analysis.event_study import run_event_study

es_metrics = [m for m in ["vessel_count", "mean_sog", "std_sog", "ais_silence_count", "cog_variance"] if m in blacksea_panel.columns]

es_results = {}
for metric in es_metrics:
    result = run_event_study(
        blacksea_panel, metric=metric, event_date=t0_date,
        estimation_window=(-90, -31), event_window=(-30, 30),
    )
    es_results[metric] = result
    sig = len(result.get("significant_dates", []))
    print(f"{metric:<25} status={result['status']:<8} significant_dates={sig}")

vessel_count              status=insufficient_estimation_data significant_dates=0
mean_sog                  status=insufficient_estimation_data significant_dates=0
std_sog                   status=insufficient_estimation_data significant_dates=0
ais_silence_count         status=insufficient_estimation_data significant_dates=0
cog_variance              status=insufficient_estimation_data significant_dates=0


---
## 8. Interrupted Time Series (ITS) Analysis

Fit segmented regression models to estimate level and slope changes after T0.

In [10]:
from mcis.analysis.its import run_its

its_results = {}
for metric in es_metrics:
    result = run_its(blacksea_panel, metric=metric, event_date=t0_date, polynomial_degree=2)
    its_results[metric] = result
    lc = result.get("level_change", {})
    sc = result.get("slope_change", {})
    l_p = lc.get("p_value", 1.0)
    s_p = sc.get("p_value", 1.0)
    print(f"{metric:<25} level_p={l_p:.3f}  slope_p={s_p:.3f}  R2={result.get('r_squared', 0):.3f}")

vessel_count              level_p=1.000  slope_p=1.000  R2=0.000
mean_sog                  level_p=1.000  slope_p=1.000  R2=0.000
std_sog                   level_p=1.000  slope_p=1.000  R2=0.000
ais_silence_count         level_p=1.000  slope_p=1.000  R2=0.000
cog_variance              level_p=1.000  slope_p=1.000  R2=0.000


---
## 9. Granger Causality

Test whether maritime metrics Granger-cause the `post_conflict` indicator.

In [11]:
from mcis.analysis.granger import run_granger

if "post_conflict" in blacksea_panel.columns:
    gc_results = {}
    for metric in es_metrics:
        result = run_granger(
            blacksea_panel,
            predictor_col=metric,
            target_col="post_conflict",
            max_lag=7,
            alpha=0.05,
        )
        gc_results[metric] = result
        best_lag = result.get("best_lag", "N/A")
        best_p = result.get("best_p_value", 1.0)
        sig_lags = result.get("significant_lags", [])
        print(f"{metric:<25} best_lag={best_lag}  best_p={best_p:.3f}  sig_lags={sig_lags}")
else:
    print("post_conflict column not found. Granger analysis requires it.")

vessel_count              best_lag=N/A  best_p=1.000  sig_lags=[]
mean_sog                  best_lag=N/A  best_p=1.000  sig_lags=[]
std_sog                   best_lag=N/A  best_p=1.000  sig_lags=[]
ais_silence_count         best_lag=N/A  best_p=1.000  sig_lags=[]
cog_variance              best_lag=N/A  best_p=1.000  sig_lags=[]


---
## 10. Difference-in-Differences (DiD)

Compare treated vs control grid cells before and after T0.

In [12]:
from mcis.analysis.did import run_did

# Use grid panel for DiD if we have multiple grid cells
if len(grid_panel) > 1:
    unique_grids = grid_panel.index.get_level_values("grid_id").unique()
    print(f"Unique grids: {len(unique_grids)}")
    if len(unique_grids) >= 4:
        grids = unique_grids.tolist()
        treated = grids[:2]
        control = grids[2:4]
        did_result = run_did(
            grid_panel,
            treated_grid_ids=treated,
            control_grid_ids=control,
            event_date=t0_date,
            metric="vessel_count",
        )
        print(f"DiD coef: {did_result.get('did_coef')}")
        print(f"DiD p-value: {did_result.get('did_p_value')}")
        print(f"Parallel trends: {did_result.get('parallel_trends')}")
    else:
        print("Not enough grid cells for DiD analysis")
else:
    print("Grid panel too small for DiD analysis")

Unique grids: 3
Not enough grid cells for DiD analysis


---
## 11. Anomaly Detection Models

Train rolling z-score, EWMA, and robust Mahalanobis detectors to identify pre-conflict behavioral anomalies.

In [14]:
from mcis.models.anomaly import (
    RollingZScoreDetector, EWMADetector, RobustMahalanobisDetector,
)
from mcis.models.evaluate import (
    compute_anomaly_metrics, run_placebo_dates,
    compute_classification_metrics,
)
from mcis.validation import assert_no_leakage

# Prepare panel with date column
panel_df = blacksea_panel.copy()
panel_df["date"] = panel_df.index
panel_df = panel_df.sort_values("date")

feature_cols = cfg.get("model", {}).get("features_to_use", [])
assert_no_leakage(feature_cols, cfg)
print(f"Features: {feature_cols}")

# Temporal split
t0 = pd.Timestamp(cfg["conflict"]["t0"])
model_cfg = cfg.get("model", {})
warning_days = model_cfg.get("early_warning_window_days", 30)

train_df = panel_df[panel_df["date"] <= pd.Timestamp("2021-12-25")].copy()
calib_df = panel_df[(panel_df["date"] >= pd.Timestamp("2021-12-26")) & (panel_df["date"] <= pd.Timestamp("2022-01-24"))].copy()
eval_df = panel_df[(panel_df["date"] >= pd.Timestamp("2022-01-25")) & (panel_df["date"] <= pd.Timestamp("2022-02-23"))].copy()
post_df = panel_df[panel_df["date"] >= t0].copy()

print(f"Train: {len(train_df)} days, Calib: {len(calib_df)} days, Eval: {len(eval_df)} days, Post: {len(post_df)} days")

if len(train_df) > 0:
    train_X = train_df[feature_cols].fillna(0)
    calib_X = calib_df[feature_cols].fillna(0) if not calib_df.empty else pd.DataFrame(columns=feature_cols)
    eval_X = eval_df[feature_cols].fillna(0) if not eval_df.empty else pd.DataFrame(columns=feature_cols)
    post_X = post_df[feature_cols].fillna(0) if not post_df.empty else pd.DataFrame(columns=feature_cols)

    detectors = {
        "rolling_zscore": RollingZScoreDetector(window=30, threshold=3.0),
        "ewma": EWMADetector(span=14, threshold=3.0),
    }

    for name, detector in detectors.items():
        print(f"\n--- {name} ---")
        try:
            detector.fit(train_X)
            train_scores = detector.predict(train_X).mean(axis=1)
            eval_scores = detector.predict(eval_X).mean(axis=1) if not eval_X.empty else pd.Series(dtype=float)
            post_scores = detector.predict(post_X).mean(axis=1) if not post_X.empty else pd.Series(dtype=float)

            if not eval_scores.empty:
                metrics_result = compute_anomaly_metrics(
                    scores=eval_scores,
                    dates=eval_df["date"],
                    event_date=cfg["conflict"]["t0"],
                    warning_window_days=warning_days,
                )
                print(f"  First alert lead: {metrics_result.get('first_alert_lead_days')} days")
                print(f"  False alarms/30d: {metrics_result.get('false_alarms_per_30_days')}")
        except Exception as e:
            print(f"  Skipped: {e}")
else:
    print("Not enough training data for anomaly detection models")

Features: ['vessel_count', 'mean_sog', 'std_sog', 'ais_silence_count', 'cog_variance']
Train: 0 days, Calib: 0 days, Eval: 1 days, Post: 2 days
Not enough training data for anomaly detection models


---
## 12. Robustness — Placebo & Multiple Testing

Run placebo cut-date tests and FDR/Holm correction for ITS p-values.

In [15]:
from mcis.analysis.robustness import run_placebo_cutdates, apply_multiple_testing

# Placebo cut-date test
placebo_offsets = [-60, -45, -30, -15, 15, 30, 45, 60]
placebo_results = {}
for metric in es_metrics[:3]:
    result = run_placebo_cutdates(
        blacksea_panel, metric=metric,
        candidate_offsets=placebo_offsets,
        estimation_window=(-90, -31), event_window=(-30, 30),
    )
    placebo_results[metric] = result
    print(f"{metric:<25} median_p={result.get('median_p_value', 'N/A')}")

# Multiple testing correction
its_pvalues = {m: its_results[m].get("level_change", {}).get("p_value", 1.0) for m in es_metrics}
its_pvalues = {k: v for k, v in its_pvalues.items() if isinstance(v, (int, float)) and np.isfinite(v)}

if its_pvalues:
    fdr_result = apply_multiple_testing(its_pvalues, alpha=0.05)
    print(f"\n{'Metric':<25} {'Raw p':<12} {'FDR-BH':<12} {'BH sig':<8} {'Holm sig':<8}")
    print("-" * 65)
    for metric, vals in fdr_result["adjusted_p_values"].items():
        print(f"{metric:<25} {vals['raw_p_value']:<12.3e} {vals['fdr_bh_p_value']:<12.3e} {vals['fdr_bh_reject']!s:<8} {vals['holm_reject']!s:<8}")

vessel_count              median_p=1.0
mean_sog                  median_p=1.0
std_sog                   median_p=1.0

Metric                    Raw p        FDR-BH       BH sig   Holm sig
-----------------------------------------------------------------
vessel_count              1.000e+00    1.000e+00    False    False   
mean_sog                  1.000e+00    1.000e+00    False    False   
std_sog                   1.000e+00    1.000e+00    False    False   
ais_silence_count         1.000e+00    1.000e+00    False    False   
cog_variance              1.000e+00    1.000e+00    False    False   


---
## 13. Forecasting Models

Fit a VAR forecasting model and compute forecast-error-based anomaly scores.

In [16]:
from mcis.models.forecasting import VARForecaster
from mcis.models.evaluate import compute_forecast_error_anomaly

# Prepare data (same as anomaly detection cell, in case that cell was skipped)
_f_panel = blacksea_panel.copy()
_f_panel["date"] = _f_panel.index
_f_panel = _f_panel.sort_values("date")
_f_features = cfg.get("model", {}).get("features_to_use", [])

_f_train_df = _f_panel[_f_panel["date"] <= pd.Timestamp("2021-12-25")].copy()
_f_eval_df = _f_panel[(_f_panel["date"] >= pd.Timestamp("2022-01-25")) & (_f_panel["date"] <= pd.Timestamp("2022-02-23"))].copy()

if len(_f_train_df) >= 19:
    _f_train_X = _f_train_df[_f_features].fillna(0)
    _f_eval_X = _f_eval_df[_f_features].fillna(0) if not _f_eval_df.empty else pd.DataFrame(columns=_f_features)

    forecaster = VARForecaster(maxlags=min(14, len(_f_train_X) // 4))
    forecaster.fit(_f_train_X)

    horizon = min(len(_f_eval_X), 14)
    if horizon >= 1 and not _f_eval_X.empty:
        y_pred = forecaster.predict(_f_train_X, horizon=horizon)
        y_true = _f_eval_X.iloc[:horizon]
        error_scores = compute_forecast_error_anomaly(y_true, y_pred, method="zscore")
        print(f"Forecast error anomaly scores: {len(error_scores)} points")
        print(f"  Mean score: {error_scores.mean():.3f}")
        print(f"  Max score: {error_scores.max():.3f}")
    else:
        print("Not enough eval data for forecasting")
else:
    print(f"Not enough training data for VAR (need >=19 rows, have {len(_f_train_df)})")


Not enough training data for VAR (need >=19 rows, have 0)


---
## 14. Model Registry & Model Card

Generate model cards and a registry dashboard comparing all model runs.

In [17]:
from mcis.models.model_card import generate_model_card
from mcis.models.registry import ModelCardRegistry

MODELS_DIR = OUTPUT_DIR / "models"
ensure_dir(MODELS_DIR)

registry = ModelCardRegistry(MODELS_DIR / "registry")

# Register a sample result if we have any
sample_result = {
    "model_name": "rolling_zscore",
    "formulation": "anomaly",
    "data_validity_mode": cfg["project"]["data_validity_mode"],
    "train_period": [str(train_df["date"].min().date()) if not train_df.empty else "N/A",
                     str(train_df["date"].max().date()) if not train_df.empty else "N/A"],
    "feature_cols": feature_cols,
    "metrics": {"n_alerts_warning_window": 0},
    "first_alert_lead_days": None,
    "placebo_p_value": None,
    "caveats": ["Single-event limitation: only one conflict onset date."],
}

registry.register_run(sample_result)
card_path = generate_model_card(sample_result, MODELS_DIR)
print(f"Model card: {card_path}")

dashboard_path = registry.generate_dashboard(FIG_DIR)
print(f"Registry dashboard: {dashboard_path}")

Model card: outputs\models\rolling_zscore_model_card_20260519_062751.md
Registry dashboard: outputs\figures\registry_dashboard_20260519_062751.md


---
## 15. Map Visualization

Generate interactive maps showing traffic density and grid anomalies.

In [18]:
try:
    from mcis.viz.maps import plot_traffic_density, plot_grid_anomaly
    _maps_ok = True
except ImportError:
    _maps_ok = False
    print("Map functions unavailable: install folium and plotly")

if _maps_ok:
    grid_reset = grid_panel.reset_index()
    needs_centroid = {"centroid_lon", "centroid_lat"} - set(grid_reset.columns)
    if needs_centroid:
        grid_df = aggregator.grid
        grid_reset = grid_reset.merge(grid_df[["grid_id", "centroid_lon", "centroid_lat"]], on="grid_id")

    try:
        m = plot_traffic_density(grid_reset.set_index(["grid_id", "time_bucket"]), FIG_DIR / "traffic_density.html", metric="vessel_count")
        print("Saved: traffic_density.html")
    except Exception as e:
        print(f"Traffic density map skipped: {e}")

    try:
        m = plot_grid_anomaly(grid_reset.set_index(["grid_id", "time_bucket"]), "vessel_count", FIG_DIR / "grid_anomaly.html", t0=t0_date)
        print("Saved: grid_anomaly.html")
    except Exception as e:
        print(f"Grid anomaly map skipped: {e}")
else:
    print("Map visualization section skipped (missing optional dependencies)")


Traffic density map skipped: folium is required for map visualizations. Install it with: pip install folium
Grid anomaly map skipped: folium is required for map visualizations. Install it with: pip install folium


---
## Summary

The MCIS pipeline completed all stages. Results are saved to:
- **Figures**: `outputs/figures/`
- **Tables**: `outputs/tables/`
- **Models**: `outputs/models/`
- **Metadata**: `outputs/metadata/`

Key findings depend on the available data window. For a full 12-month analysis, use the primary file `ais_blacksea_3y.csv`.